# FiiCode 2026 Final — CatBoost Safe Lag Model

Goal: improve over the baseline public score **2.5280** and over the base CatBoost model.

This notebook trains a **non-recursive** CatBoost model with:

- categorical identity features;
- calendar features;
- price features;
- historical mean features;
- safe weekly lag features;
- safe rolling mean features.

Important design:
- no `lag_1`, `lag_2`, `lag_3` yet;
- validation demand is hidden before lag creation;
- test demand is `NaN` before lag creation;
- this avoids future leakage.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## 2. Experiment switches

In [ ]:
BASELINE_PUBLIC_SCORE = 2.5280

USE_MEAN_FEATURES = True
USE_PRICE_FEATURES = True
USE_LAG_FEATURES = True
USE_ROLLING_FEATURES = False

USE_LOG_TARGET = False
ROUND_PREDICTIONS = False

RANDOM_SEED = 42

# Faster competition config.
CATBOOST_ITERATIONS = 1500
CATBOOST_LEARNING_RATE = 0.05
CATBOOST_DEPTH = 8
CATBOOST_L2 = 5
CATBOOST_EARLY_STOPPING = 100

SAFE_LAGS = [7, 14, 21, 28]
ROLL_WINDOWS = [7, 14, 28]

# plan.in alignment:
# False -> step 04: lags only
# True  -> step 05: lags + rolling means
if USE_LAG_FEATURES and USE_ROLLING_FEATURES:
    RUN_LABEL = "catboost_lags_rolling"
    OUTPUT_FILE = "submission_catboost_lags_rolling.csv"
elif USE_LAG_FEATURES:
    RUN_LABEL = "catboost_safe_lags"
    OUTPUT_FILE = "submission_catboost_safe_lags.csv"
else:
    raise ValueError("This notebook expects USE_LAG_FEATURES = True.")

SUBMISSION_FILE = "submission.csv"

print("RUN_LABEL:", RUN_LABEL)
print("OUTPUT_FILE:", OUTPUT_FILE)
print("SUBMISSION_FILE:", SUBMISSION_FILE)

## 3. Find input files automatically

In [ ]:
train_path = None
test_path = None
sample_path = None

for dirname, _, filenames in os.walk("/kaggle/input"):
    if "train_final.csv" in filenames:
        train_path = os.path.join(dirname, "train_final.csv")
    if "test_final.csv" in filenames:
        test_path = os.path.join(dirname, "test_final.csv")
    if "sample_submission.csv" in filenames:
        sample_path = os.path.join(dirname, "sample_submission.csv")

print("train_path:", train_path)
print("test_path:", test_path)
print("sample_path:", sample_path)

if train_path is None:
    raise FileNotFoundError("Could not find train_final.csv")
if test_path is None:
    raise FileNotFoundError("Could not find test_final.csv")

## 4. Load data

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train date range:", train["date"].min(), "to", train["date"].max())
print("Test date range:", test["date"].min(), "to", test["date"].max())

display(train.head())
display(test.head())

## 5. Calendar features with shared origin

In [ ]:
ORIGIN_DATE = train["date"].min()

def add_calendar_features(df):
    df = df.copy()
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    df["day"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["days_from_start"] = (df["date"] - ORIGIN_DATE).dt.days
    return df

train = add_calendar_features(train)
test = add_calendar_features(test)

display(train[["date", "dayofweek", "is_weekend", "day", "month", "weekofyear", "days_from_start"]].head())
display(test[["date", "dayofweek", "is_weekend", "day", "month", "weekofyear", "days_from_start"]].head())

## 6. Validation split: last 14 days

In [ ]:
last_train_date = train["date"].max()
valid_start = last_train_date - pd.Timedelta(days=13)

tr_raw = train[train["date"] < valid_start].copy()
va_raw = train[train["date"] >= valid_start].copy()

print("Training period:", tr_raw["date"].min(), "to", tr_raw["date"].max())
print("Validation period:", va_raw["date"].min(), "to", va_raw["date"].max())
print("tr_raw shape:", tr_raw.shape)
print("va_raw shape:", va_raw.shape)

## 7. Helper functions: mean features

For validation, these tables are built only from the training split.

For final test, they are built from full train.

In [ ]:
def build_mean_tables(source_df):
    tables = {}

    tables["product_store_mean"] = (
        source_df.groupby(["store_id", "product_id"])["demand"]
        .mean()
        .rename("product_store_mean")
        .reset_index()
    )

    tables["product_mean"] = (
        source_df.groupby("product_id")["demand"]
        .mean()
        .rename("product_mean")
        .reset_index()
    )

    tables["store_mean"] = (
        source_df.groupby("store_id")["demand"]
        .mean()
        .rename("store_mean")
        .reset_index()
    )

    tables["category_mean"] = (
        source_df.groupby("category")["demand"]
        .mean()
        .rename("category_mean")
        .reset_index()
    )

    tables["category_store_mean"] = (
        source_df.groupby(["store_id", "category"])["demand"]
        .mean()
        .rename("category_store_mean")
        .reset_index()
    )

    tables["product_weekday_mean"] = (
        source_df.groupby(["product_id", "dayofweek"])["demand"]
        .mean()
        .rename("product_weekday_mean")
        .reset_index()
    )

    tables["product_store_weekday_mean"] = (
        source_df.groupby(["store_id", "product_id", "dayofweek"])["demand"]
        .mean()
        .rename("product_store_weekday_mean")
        .reset_index()
    )

    return tables


def add_mean_features(df, tables, global_mean):
    df = df.copy()

    df = df.merge(tables["product_store_mean"], on=["store_id", "product_id"], how="left")
    df = df.merge(tables["product_mean"], on="product_id", how="left")
    df = df.merge(tables["store_mean"], on="store_id", how="left")
    df = df.merge(tables["category_mean"], on="category", how="left")
    df = df.merge(tables["category_store_mean"], on=["store_id", "category"], how="left")
    df = df.merge(tables["product_weekday_mean"], on=["product_id", "dayofweek"], how="left")
    df = df.merge(tables["product_store_weekday_mean"], on=["store_id", "product_id", "dayofweek"], how="left")

    mean_cols = [
        "product_store_mean",
        "product_mean",
        "store_mean",
        "category_mean",
        "category_store_mean",
        "product_weekday_mean",
        "product_store_weekday_mean"
    ]

    for c in mean_cols:
        df[c] = df[c].fillna(global_mean)

    return df

## 8. Helper functions: price features

In [ ]:
def build_price_tables(source_df):
    tables = {}

    tables["product_avg_price"] = (
        source_df.groupby("product_id")["price"]
        .mean()
        .rename("product_avg_price")
        .reset_index()
    )

    tables["category_avg_price"] = (
        source_df.groupby("category")["price"]
        .mean()
        .rename("category_avg_price")
        .reset_index()
    )

    return tables


def add_price_features(df, tables):
    df = df.copy()
    df["price_missing"] = df["price"].isna().astype(int)

    df = df.merge(tables["product_avg_price"], on="product_id", how="left")
    df = df.merge(tables["category_avg_price"], on="category", how="left")

    df["product_avg_price"] = df["product_avg_price"].fillna(df["price"])
    df["category_avg_price"] = df["category_avg_price"].fillna(df["price"])

    df["price_vs_product_avg"] = df["price"] / df["product_avg_price"].replace(0, np.nan)
    df["price_vs_category_avg"] = df["price"] / df["category_avg_price"].replace(0, np.nan)

    df["price_diff_product_avg"] = df["price"] - df["product_avg_price"]
    df["price_diff_category_avg"] = df["price"] - df["category_avg_price"]

    for c in [
        "price_vs_product_avg",
        "price_vs_category_avg",
        "price_diff_product_avg",
        "price_diff_category_avg"
    ]:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df

## 9. Helper functions: safe lag and rolling features

Key anti-leakage rule:

For validation, validation demand is set to `NaN` before lag creation.  
For test, test demand is already unknown/`NaN`.

So lag/rolling features cannot see future target values.

In [ ]:
def add_lag_rolling_features(combined_df, group_cols=["store_id", "product_id"]):
    df = combined_df.copy()
    df = df.sort_values(group_cols + ["date"]).reset_index(drop=True)

    lag_cols = []
    roll_cols = []

    if USE_LAG_FEATURES:
        for lag in SAFE_LAGS:
            col = f"lag_{lag}"
            df[col] = df.groupby(group_cols)["demand"].shift(lag)
            lag_cols.append(col)

    if USE_ROLLING_FEATURES:
        for window in ROLL_WINDOWS:
            col = f"roll_mean_{window}"
            df[col] = (
                df.groupby(group_cols)["demand"]
                  .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            )
            roll_cols.append(col)

    return df, lag_cols, roll_cols


def fill_lag_roll_missing(df, lag_cols, roll_cols, global_mean):
    df = df.copy()
    fill_cols = lag_cols + roll_cols

    for c in fill_cols:
        df[c] = df[c].fillna(df.get("product_store_mean"))
        df[c] = df[c].fillna(df.get("product_mean"))
        df[c] = df[c].fillna(df.get("category_store_mean"))
        df[c] = df[c].fillna(df.get("category_mean"))
        df[c] = df[c].fillna(global_mean)

    return df

## 10. Build leakage-safe validation features

Process:
1. Keep real demand in training split.
2. Copy validation split but hide its demand as `NaN`.
3. Concatenate training + hidden-validation.
4. Create lag and rolling features.
5. Split back into train/validation rows.
6. Restore validation target separately from `va_raw`.

In [ ]:
tr_marker = tr_raw.copy()
tr_marker["part"] = "train"

va_marker = va_raw.copy()
va_marker["part"] = "valid"
va_marker["true_demand"] = va_marker["demand"]
va_marker["demand"] = np.nan

valid_combined = pd.concat([tr_marker, va_marker], axis=0, ignore_index=True)

# Mean and price features from training split only.
tr_global_mean = tr_raw["demand"].mean()

if USE_MEAN_FEATURES:
    tr_mean_tables = build_mean_tables(tr_raw)
    valid_combined = add_mean_features(valid_combined, tr_mean_tables, tr_global_mean)

if USE_PRICE_FEATURES:
    tr_price_tables = build_price_tables(tr_raw)
    valid_combined = add_price_features(valid_combined, tr_price_tables)

valid_combined, lag_cols, roll_cols = add_lag_rolling_features(valid_combined)
valid_combined = fill_lag_roll_missing(valid_combined, lag_cols, roll_cols, tr_global_mean)

tr_fe = valid_combined[valid_combined["part"] == "train"].copy()
va_fe = valid_combined[valid_combined["part"] == "valid"].copy()

# The model target for validation is stored separately.
va_fe["demand"] = va_fe["true_demand"]

# Remove helper target column after restoration.
if "true_demand" in tr_fe.columns:
    tr_fe = tr_fe.drop(columns=["true_demand"])
if "true_demand" in va_fe.columns:
    va_fe = va_fe.drop(columns=["true_demand"])

print("Lag columns:", lag_cols)
print("Rolling columns:", roll_cols)
print("tr_fe shape:", tr_fe.shape)
print("va_fe shape:", va_fe.shape)

display(va_fe[["date", "store_id", "product_id", "demand"] + lag_cols + roll_cols].head(20))

## 11. Check missing lag/rolling values

In [ ]:
check_cols = lag_cols + roll_cols
print("Missing values after fill:")
display(tr_fe[check_cols].isna().sum())
display(va_fe[check_cols].isna().sum())

## 12. Prepare categorical columns

In [ ]:
cat_cols = [
    "store_id",
    "city",
    "region",
    "product_id",
    "product_name",
    "category",
    "holiday_name"
]

cat_cols = [c for c in cat_cols if c in tr_fe.columns]

for df in [tr_fe, va_fe]:
    for c in cat_cols:
        df[c] = df[c].fillna("missing").astype(str)

print("Categorical columns:", cat_cols)

## 13. Feature selection

In [ ]:
drop_cols = [
    "row_ID",
    "row_id",
    "date",
    "demand",
    "part"
]

features = [c for c in tr_fe.columns if c not in drop_cols]
cat_feature_indices = [features.index(c) for c in cat_cols if c in features]

print("Number of features:", len(features))
print(features)
print("Categorical feature indices:", cat_feature_indices)

## 14. Train CatBoost lag model

In [ ]:
X_tr = tr_fe[features]
y_tr = tr_fe["demand"]

X_va = va_fe[features]
y_va = va_fe["demand"]

if USE_LOG_TARGET:
    y_train_model = np.log1p(y_tr)
    y_valid_model = np.log1p(y_va)
else:
    y_train_model = y_tr
    y_valid_model = y_va

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=CATBOOST_ITERATIONS,
    learning_rate=CATBOOST_LEARNING_RATE,
    depth=CATBOOST_DEPTH,
    l2_leaf_reg=CATBOOST_L2,
    random_seed=RANDOM_SEED,
    early_stopping_rounds=CATBOOST_EARLY_STOPPING,
    verbose=100
)

model.fit(
    X_tr,
    y_train_model,
    eval_set=(X_va, y_valid_model),
    cat_features=cat_feature_indices
)

## 15. Validation RMSE

In [ ]:
va_pred = model.predict(X_va)

if USE_LOG_TARGET:
    va_pred = np.expm1(va_pred)

va_pred = np.clip(va_pred, 0, None)

rmse = rmse_score(y_va, va_pred)

print("Safe lag validation RMSE:", rmse)
print("Baseline public score anchor:", BASELINE_PUBLIC_SCORE)
print("Best iteration:", model.get_best_iteration())

preview = va_fe[["date", "store_id", "product_id", "category", "demand"]].copy()
preview["prediction"] = va_pred
preview["error"] = preview["prediction"] - preview["demand"]

display(preview.head(30))
display(preview["error"].describe())

## 16. Feature importance

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(importance.head(40))

## 17. Build final full-train + test features

For final submission:
1. full train has true demand;
2. test gets `demand = NaN`;
3. concat;
4. create non-recursive lag/rolling features;
5. predict all test rows.

In [ ]:
full_train = train.copy()
full_train["part"] = "train"

full_test = test.copy()
full_test["part"] = "test"
full_test["demand"] = np.nan

full_combined = pd.concat([full_train, full_test], axis=0, ignore_index=True)

full_global_mean = full_train["demand"].mean()

if USE_MEAN_FEATURES:
    full_mean_tables = build_mean_tables(full_train)
    full_combined = add_mean_features(full_combined, full_mean_tables, full_global_mean)

if USE_PRICE_FEATURES:
    full_price_tables = build_price_tables(full_train)
    full_combined = add_price_features(full_combined, full_price_tables)

full_combined, final_lag_cols, final_roll_cols = add_lag_rolling_features(full_combined)
full_combined = fill_lag_roll_missing(full_combined, final_lag_cols, final_roll_cols, full_global_mean)

full_train_fe = full_combined[full_combined["part"] == "train"].copy()
full_test_fe = full_combined[full_combined["part"] == "test"].copy()

for df in [full_train_fe, full_test_fe]:
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].fillna("missing").astype(str)

features_final = [c for c in full_train_fe.columns if c not in drop_cols]
cat_feature_indices_final = [features_final.index(c) for c in cat_cols if c in features_final]

print("Final train features shape:", full_train_fe[features_final].shape)
print("Final test features shape:", full_test_fe[features_final].shape)
print("Final number of features:", len(features_final))
print(features_final)

print("Missing values in final test lag/rolling:")
display(full_test_fe[final_lag_cols + final_roll_cols].isna().sum())

## 18. Retrain final model

In [ ]:
best_iter = model.get_best_iteration()

if best_iter is None or best_iter < 100:
    final_iterations = CATBOOST_ITERATIONS
else:
    final_iterations = best_iter + 1

print("Final iterations:", final_iterations)

X_full = full_train_fe[features_final]
y_full = full_train_fe["demand"]

if USE_LOG_TARGET:
    y_full_model = np.log1p(y_full)
else:
    y_full_model = y_full

final_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=final_iterations,
    learning_rate=CATBOOST_LEARNING_RATE,
    depth=CATBOOST_DEPTH,
    l2_leaf_reg=CATBOOST_L2,
    random_seed=RANDOM_SEED,
    verbose=100
)

final_model.fit(
    X_full,
    y_full_model,
    cat_features=cat_feature_indices_final
)

## 19. Predict test

In [ ]:
test_predictions = final_model.predict(full_test_fe[features_final])

if USE_LOG_TARGET:
    test_predictions = np.expm1(test_predictions)

test_predictions = np.clip(test_predictions, 0, None)

if ROUND_PREDICTIONS:
    test_predictions = np.rint(test_predictions).astype(int)

print("Prediction stats:")
print(pd.Series(test_predictions).describe())

## 20. Create submission

In [ ]:
if "row_id" in full_test_fe.columns:
    row_col = "row_id"
elif "row_ID" in full_test_fe.columns:
    row_col = "row_ID"
else:
    raise ValueError("Could not find row_id or row_ID in test data.")

submission = pd.DataFrame({
    "row_id": full_test_fe[row_col],
    "demand": test_predictions
})

submission.to_csv(OUTPUT_FILE, index=False)
submission.to_csv(SUBMISSION_FILE, index=False)

display(submission.head())
print(f"Saved {OUTPUT_FILE}")
print(f"Saved {SUBMISSION_FILE}")

## 21. Result log

Record:

- validation RMSE;
- public score;
- whether it beats 2.5280;
- whether it beats base CatBoost.

Next if this is good:
- recursive safe lags.

Next if this is bad:
- try lags only without rolling means;
- try product-store-weekday average baseline;
- inspect missing lag feature importance.